In [2]:
"""

@@ PDF paser도 좋은거 골라야 함


pdf_text_extractor.py
────────────────────────────────────────────────
• data/            ⟶ PDF 원본이 들어있는 루트 폴더
• extracted_texts/ ⟶ PDF별 추출 결과(.txt) 저장 위치
────────────────────────────────────────────────
pip install pymupdf tqdm
"""

from pathlib import Path
import fitz           # PyMuPDF
import json
from tqdm import tqdm  # 진행률 표시용 (선택)

ROOT_PDF_DIR   = Path("data")
OUTPUT_TXT_DIR = Path("extracted_texts")
META_JSON_PATH = Path("extracted_texts/_extraction_meta.json")  # 진행 내역‧검증용

OUTPUT_TXT_DIR.mkdir(parents=True, exist_ok=True)

def extract_text_from_pdf(pdf_path: Path) -> str:
    """한 PDF(멀티 페이지)의 텍스트를 전부 이어붙여 반환"""
    doc  = fitz.open(pdf_path)
    text = []
    for page in doc:                        # 페이지 순회
        page_text = page.get_text("text")   # layout 無, 순수 텍스트
        text.append(page_text.strip())
    return "\n\n".join(text)

def main():
    # 이전에 완료한 PDF는 건너뛰기 위해 메타 파일 로드
    done_files = {}
    if META_JSON_PATH.exists():
        done_files = json.loads(META_JSON_PATH.read_text(encoding="utf-8"))

    new_meta = {}

    pdf_paths = list(ROOT_PDF_DIR.rglob("*.pdf"))
    if not pdf_paths:
        print("처리할 PDF가 없습니다.")
        return

    for pdf_path in tqdm(pdf_paths, desc="PDF 전처리"):
        pdf_rel  = pdf_path.relative_to(ROOT_PDF_DIR)             # data 하위 상대경로
        txt_path = OUTPUT_TXT_DIR / pdf_rel.with_suffix(".txt")   # .txt 경로 매핑

        # 이미 추출 완료된 파일이면 건너뜀
        if str(pdf_rel) in done_files and txt_path.exists():
            new_meta[str(pdf_rel)] = done_files[str(pdf_rel)]
            continue

        # 추출
        try:
            pdf_text = extract_text_from_pdf(pdf_path)
            txt_path.parent.mkdir(parents=True, exist_ok=True)
            txt_path.write_text(pdf_text, encoding="utf-8")

            # 간단 검증: 글자 수/줄 수 저장
            lines = pdf_text.splitlines()
            info = {
                "chars": len(pdf_text),
                "lines": len(lines),
                "preview": pdf_text[:200].replace("\n", " ") + "…"  # 앞 200자
            }
            new_meta[str(pdf_rel)] = info

            # 콘솔에도 일부 확인
            print(f"\n✅ [{pdf_rel}] → {info['chars']} chars, {info['lines']} lines")
            print(f"   preview: {info['preview']}\n")

        except Exception as e:
            print(f"❌ [{pdf_rel}] 추출 실패: {e}")

    # 메타정보 저장(누적)
    META_JSON_PATH.write_text(json.dumps(new_meta, ensure_ascii=False, indent=2),
                              encoding="utf-8")
    print("\n📝 전처리 완료 – 결과는 extracted_texts/ 폴더와 _extraction_meta.json을 확인하세요.")


In [3]:

if __name__ == "__main__":
    main()


PDF 전처리: 100%|██████████| 30/30 [00:00<00:00, 16617.69it/s]


📝 전처리 완료 – 결과는 extracted_texts/ 폴더와 _extraction_meta.json을 확인하세요.


In [ ]:
"""
백터 DB 구축
!pip install faiss-cpu
! pip install sentence_transformers
! pip install --upgrade --force-reinstall sentence-transformers
! pip install pandas
! pip install pyarrow
! pip install dill
! pip install aiohttp
! pip install numpy
! pip isntall faiss
! pip install accelerate
"""



  Using cached accelerate-1.8.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.8.1-py3-none-any.whl (365 kB)


In [1]:
#!/usr/bin/env python3
# faiss_ingest_chunked.py

"""
PDF에서 추출된 텍스트(.txt)를 최대 512토큰 청크로 분할하여
임베딩하고 FAISS에 저장하는 스크립트입니다.
"""

import json
from pathlib import Path

import numpy as np
import faiss
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ─────────── 설정 ───────────
MODEL_DIR   = "/home/조기정/project/RAG_LLM/src/test/embedding"
TEXT_DIR    = Path("extracted_texts")
INDEX_PATH  = Path("faiss_index.idx")
META_PATH   = Path("faiss_metadata.json")

MAX_TOKENS  = 512   # 청크당 최대 토큰 수
OVERLAP     = 64    # 청크 간 중복 토큰 수
# ──────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) 토크나이저·모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model     = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True)
model.to(device).eval()

# 2) mean pooling 정의
def mean_pooling(outputs, attention_mask):
    embeddings = outputs.last_hidden_state  # (batch, seq_len, dim)
    mask       = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
    summed     = torch.sum(embeddings * mask, dim=1)
    counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts  # (batch, dim)

# 3) 텍스트를 토큰 기반으로 청크 분할
def chunk_text(text: str, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    input_ids = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt")[0]
    total_len = input_ids.size(0)
    chunks = []
    start = 0
    while start < total_len:
        end = min(start + max_tokens, total_len)
        chunk_ids = input_ids[start:end]
        chunk_txt = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_txt)
        start += max_tokens - overlap
    return chunks

# 4) FAISS 인덱스 준비
emb_dim   = model.config.hidden_size
cpu_index = faiss.IndexFlatIP(emb_dim)
index     = faiss.IndexIDMap2(cpu_index)

# 5) 기존 메타데이터 로드 및 next_id 계산
if META_PATH.exists():
    metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
    next_id  = max(map(int, metadata.keys())) + 1
else:
    metadata = {}
    next_id  = 0

# 6) TXT 파일 순회 → 청크 임베딩 → FAISS 추가
for txt_path in TEXT_DIR.rglob("*.txt"):
    rel_path = str(txt_path.relative_to(TEXT_DIR))
    # 이미 처리된 문서 건너뛰기
    if any(info.get("path") == rel_path for info in metadata.values()):
        continue

    text = txt_path.read_text(encoding="utf-8")
    chunks = chunk_text(text)

    for chunk_idx, chunk in enumerate(chunks):
        # 토크나이즈 및 모델 통과
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_TOKENS,
            padding="longest"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # 풀링 + 정규화
        emb = mean_pooling(outputs, inputs["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)

        vec = emb.cpu().numpy().astype("float32")
        idx = next_id
        index.add_with_ids(vec, np.array([idx], dtype="int64"))

        # 메타데이터에 경로 + 청크 인덱스 저장
        metadata[str(idx)] = {
            "path": rel_path,
            "chunk": chunk_idx
        }
        next_id += 1

        print(f"Added ID={idx} | {rel_path} [chunk {chunk_idx}]")

# 7) FAISS 인덱스·메타데이터 시디스크 저장
faiss.write_index(index, str(INDEX_PATH))
with META_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ FAISS index and metadata saved.")


/home/조기정/.conda/envs/Qwen2.5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.74it/s]


Added ID=0 | 보수규정_20250228.txt [chunk 0]
Added ID=1 | 보수규정_20250228.txt [chunk 1]
Added ID=2 | 보수규정_20250228.txt [chunk 2]
Added ID=3 | 보수규정_20250228.txt [chunk 3]
Added ID=4 | 보수규정_20250228.txt [chunk 4]
Added ID=5 | 보수규정_20250228.txt [chunk 5]
Added ID=6 | 보수규정_20250228.txt [chunk 6]
Added ID=7 | 보수규정_20250228.txt [chunk 7]
Added ID=8 | 보수규정_20250228.txt [chunk 8]
Added ID=9 | 보수규정_20250228.txt [chunk 9]
Added ID=10 | 보수규정_20250228.txt [chunk 10]
Added ID=11 | 보수규정_20250228.txt [chunk 11]
Added ID=12 | 보수규정_20250228.txt [chunk 12]
Added ID=13 | 보수규정_20250228.txt [chunk 13]
Added ID=14 | 보수규정_20250228.txt [chunk 14]
Added ID=15 | 보수규정_20250228.txt [chunk 15]
Added ID=16 | 보수규정_20250228.txt [chunk 16]
Added ID=17 | 보수규정_20250228.txt [chunk 17]
Added ID=18 | 보수규정_20250228.txt [chunk 18]
Added ID=19 | 보수규정_20250228.txt [chunk 19]
Added ID=20 | 보수규정_20250228.txt [chunk 20]
Added ID=21 | 보수규정_20250228.txt [chunk 21]
Added ID=22 | 보수규정_20250228.txt [chunk 22]
Added ID=23 | 보수규정_20250228.txt

In [ ]:
"""#!/usr/bin/env python3
# faiss_search_chunked.py

백터 DB 검색 후 텍스트 스니펫 반환



사용자 쿼리를 임베딩하여 FAISS에서 유사도를 기준으로 문서 청크를 검색하고,
해당 청크의 텍스트를 함께 반환하는 스크립트입니다.


import argparse
import json
from pathlib import Path

import numpy as np
import faiss
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ─────────── 설정 ───────────
MODEL_DIR   = "/home/조기정/project/RAG_LLM/src/test/embedding"
TEXT_DIR    = Path("extracted_texts")
INDEX_PATH  = Path("faiss_index.idx")
META_PATH   = Path("faiss_metadata.json")

MAX_TOKENS  = 512   # 청크당 최대 토큰 수 (ingest와 동일하게 설정)
OVERLAP     = 64    # 청크 간 중복 토큰 수 (ingest와 동일하게 설정)
# ──────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) 토크나이저·모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model     = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True)
model.to(device).eval()

# 2) mean pooling 정의
def mean_pooling(outputs, attention_mask):
    token_embeddings = outputs.last_hidden_state  # (batch, seq_len, dim)
    mask_expanded    = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed           = torch.sum(token_embeddings * mask_expanded, dim=1)
    counts           = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    return summed / counts  # (batch, dim)

# 3) 청크 분할 함수 (ingest와 동일)
def chunk_text(text: str, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    input_ids = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt")[0]
    total_len = input_ids.size(0)
    chunks = []
    start = 0
    while start < total_len:
        end = min(start + max_tokens, total_len)
        chunk_ids = input_ids[start:end]
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        start += max_tokens - overlap
    return chunks

# 4) FAISS 인덱스·메타데이터 로드
if not INDEX_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError("faiss_index.idx 또는 faiss_metadata.json 파일을 찾을 수 없습니다.")

index    = faiss.read_index(str(INDEX_PATH))
with META_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

# 5) 검색 함수
def search(query: str, top_k: int = 5):
    # 5.1) 쿼리 임베딩
    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_TOKENS,
        padding="longest"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    q_emb = mean_pooling(outputs, inputs["attention_mask"])
    q_emb = F.normalize(q_emb, p=2, dim=1)
    q_vec = q_emb.cpu().numpy().astype("float32")

    # 5.2) FAISS 검색
    distances, indices = index.search(q_vec, top_k)

    # 5.3) 결과 조립
    results = []
    for score, idx in zip(distances[0], indices[0]):
        if idx < 0:
            continue
        info      = metadata.get(str(idx), {})
        rel_path  = info.get("path")
        chunk_idx = info.get("chunk")

        # 청크 텍스트 로드
        txt_path = TEXT_DIR / rel_path
        full_text = txt_path.read_text(encoding="utf-8")
        chunks = chunk_text(full_text)
        snippet = chunks[chunk_idx] if chunk_idx < len(chunks) else ""

        results.append({
            "id":         idx,
            "score":      float(score),
            "path":       rel_path,
            "chunk":      chunk_idx,
            "text_snippet": snippet
        })

    return results

# 6) CLI
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description=""
    )
    parser.add_argument("query", type=str, help="검색할 문장 또는 쿼리")
    parser.add_argument("--top_k", type=int, default=5, help="가져올 상위 k개 결과")
    args = parser.parse_args()

    hits = search(args.query, args.top_k)
    if not hits:
        print("▶ 검색 결과가 없습니다.")
    else:
        print(f"▶ Top {len(hits)} results for \"{args.query}\":\n")
        for r in hits:
            print(f"ID {r['id']} | score={r['score']:.4f} | path={r['path']} | chunk={r['chunk']}")
            print(f"snippet:\n{r['text_snippet']}\n{'─'*60}")
"""

/home/조기정/.conda/envs/Qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.02it/s]
usage: ipykernel_launcher.py [-h] [--top_k TOP_K] query
ipykernel_launcher.py: error: the following arguments are required: query


SystemExit: 2

/home/조기정/.conda/envs/Qwen/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
"USER : “인사위원회 회의록 초안을 작성해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제6조(인사위원회 및 추천심사위원회<2005.6.30>) 전문입니다.
“① 인사관리의 합리화를 기하기 위하여 인사위원회를 둔다.
② 직원승진과 내부직위공모 등을 위하여 필요한 경우, 선발의 타당성과 공정성 확보를 위해 공적 등을 심사하는 추천심사위원회를 둔다.(2005.6.30 신설)
③ 인사위원회 및 추천심사위원회의 운영에 관하여 필요한 사항은 내규로 정한다.(2005.6.30)”

위 내용을 바탕으로, USER의 요청에 따라서 인사위원회 회의록 초안을 작성해주세요."	"USER : “채용방법 안내 문서를 만들어줘.”
RAG :
다음은 서울시설공단 인사규정 중 제8조(채용방법) 전문입니다.
“① 직원의 신규채용은 공개경쟁시험에 의한다. 다만, 다음 각호의 1에 해당하는 경우에는 경력경쟁시험에 의해 채용할 수 있다.

공개경쟁시험에 의하여 임용하는 것이 부적당한 경우에 임용예정 직무수행에 필요한 당해 분야의 국가가 인정하는 자격증 소지자를 임용하는 경우

공개경쟁시험에 의하여 결원보충이 곤란한 특수직렬의 직원을 임용하는 경우

(삭제 2003.11.7)

다른 법령에 정하여진 바에 따라 임용하는 경우

정원의 개폐 또는 예산의 감소 등에 의하여 폐직 또는 감원으로 퇴직하거나 신체·정신상의 장애로 인하여 휴직기간 만료로 퇴직한 날로부터 2년 이내에 퇴직 당시 직급 또는 그 하위 직급에 재임용하는 경우

긴급충원이 불가피한 경우

(삭제 2013.12.9)

국가 및 지방자치단체의 공무원과 그 투자기관에 재직하는 자를 채용하는 경우 (개정 ’03.11.7, 2011.4.1)”

위 내용을 바탕으로, USER의 요청에 따라서 채용방법 안내 문서를 만들어주세요.
"	"USER : “인사규정 제21조부터 제24조까지 요약해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제21조~제24조 전문입니다.
[제21조]
“① 승진은 동일직렬의 차하위 직급에서 인사위원회, 추천심사위원회 등을 통한 심사승진함을 원칙으로 실시하되 1회에 1직급, 승급은 1년에 1호봉을 원칙으로 한다. 다만 승진은 3월 중 실시하되 필요할 경우에는 그 시기를 조정하여 시행할 수 있다.(2005.6.30, 2010.6.1)USER : “인사규정 제21조부터 제24조까지 요약해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제21조~제24조 전문입니다.
[제21조]
“① 승진은 동일직렬의 차하위 직급에서 인사위원회, 추천심사위원회 등을 통한 심사승진함을 원칙으로 실시하되 1회에 1직급, 승급은 1년에 1호봉을 원칙으로 한다. 다만 승진은 3월 중 실시하되 필요할 경우에는 그 시기를 조정하여 시행할 수 있다.(2005.6.30, 2010.6.1)
② 제1항의 규정에 의한 승진의 방법 및 기준은 내규로 정한다.(2004.8.26, 2005.6.30, 2010.6.1)
③ 8급 직원이 입사 후 승진최소소요연수의 3배수(6년)이 경과하여도 승진하지 못한 경우에는 인사위원회의 심의를 거쳐 1회에 한하여 1직급(7급(보))을 근속승진시키며, 시기는 매분기 초로 한다.<신설 2019.4.19.>”

[제22조]
“① 직원을 승진시키고자 하는 경우에는 당해 직급의 승진후보자 명부의 서열을 참작하여 승진하고자 하는 결원 범위 내에서 실시한다.
② 승진후보순위의 결정은 근무성적평정, 경력평정, 교육훈련평정, 가감평정 등에 의한다. 단, 경영혁신에 현저한 공적이 있으며 그 공적이 심사위원회 심의에서 인정된 직원에 대하여 이사장은 그 직원이 해당 직급에 재임하는 동안에 승진후보순위 결정 시, 서열명부상 총점의 3% 이내의 점수를 별도로 정하여 가산할 수 있다.(2004.8.26, ’06.1.9 단서 신설)
③ 제2항의 규정에 의한 절차, 방법 및 세부사항은 내규로 정한다.”

[제23조]
“① 직원이 승진함에 있어서는 다음 각호의 기간 이상을 당해 직급에 근무하여야 한다.('99.7.2, 2010.6.1)

2급 내지 4급: 3년 이상

5급 내지 6급: 2년 6개월 이상(개정 2014.4.1)

7급 내지 8급: 2년 이상(신설 2014.4.1)<개정 2019.4.19.>

삭제<2019.4.19.>
② 삭제<'99.7.2>
③ 제1항의 직급별 승진 소요년수 산정은 휴직기간, 직위해제기간, 정직기간, 강등에 따라 직무에 종사하지 아니한 기간을 포함하지 아니한다. 다만, 제33조 제3호·제6호·제9호의2의 휴직기간은 예외로 하고, 제7호의 경우에는 휴직기간 최초 1년만을, 제10호의 경우에는 휴직기간의 50%를 근무년수에 산입하며, 징계처분 또는 직위해제처분을 받은 자가 처분 사유가 된 사건에 대해 법원 판결 또는 노동위원회 결정으로 무죄·무효·취소된 경우 그 기간을 근무년수에 산입한다.”

[제24조]
“① 징계등 처분요구, 징계등 의결요구, 징계등 처분, 직위해제 및 휴직기간 중에 있는 직원은 승진 및 승급할 수 없다. 단, 업무상 부상 또는 질병으로 판정받아 휴직 중인 자나 육아휴직자의 승급은 예외로 한다.(’03.11.7 본항개정, 2018.5.21.)”

위 내용을 바탕으로, USER의 요청에 따라서 인사규정 제21조~제24조를 요약해주세요.
② 제1항의 규정에 의한 승진의 방법 및 기준은 내규로 정한다.(2004.8.26, 2005.6.30, 2010.6.1)
③ 8급 직원이 입사 후 승진최소소요연수의 3배수(6년)이 경과하여도 승진하지 못한 경우에는 인사위원회의 심의를 거쳐 1회에 한하여 1직급(7급(보))을 근속승진시키며, 시기는 매분기 초로 한다.<신설 2019.4.19.>”
"	"USER : “인사규정 제33조(휴직사유와 기간)만 요약해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제33조(휴직사유와 기간) 전문입니다.
“① 직원이 다음 각호의 1에 해당할 때에는 휴직을 명할 수 있다.

신체·정신상의 장애로 장기요양을 요할 때(불임·난임 치료 포함): 1년 이내, 부득이한 경우 1년 연장 가능(2005.6.30, 2015.3.31.)

병역법 등 법령에 의해 징집 또는 소집된 경우: 징집·소집·동원 기간(2005.6.30)

업무로 인한 질병 또는 공상으로 1개월 이상 장기요양을 요할 때: 3년 이내(2005.6.30, 2015.3.31.)

천재지변·전시사변 등으로 생사 또는 소재 불명 시: 6개월 이내(2005.6.30)

삭제(2005.6.30)

직무 관련 분야 해외유학: 2년 이내(2005.6.30, 2015.3.31.)

임신 중 모성 보호 또는 만 8세 이하·초등2학년 이하 자녀 양육 위해 육아휴직: 3년(횟수 구분 없이 분할 사용 가능)

공단 지정 연구·교육기관 연수: 3년 이내(2005.6.30 신설)

부모·배우자·자녀·배우자 부모 간호 시: 1년 이내(본호신설 ’07.4.11)
9의2. 남녀고용평등법 제22조의2 가족돌봄휴직: 연간 90일 이내(분할 사용 시 30일 이상 단위, 무급휴가 포함)

개인적 필요의 해외유학: 3년 이내, 부득이한 경우 2년 이내 연장 가능<신설 ’09.3.1, 개정 2016.2.12>”

위 내용을 바탕으로, USER의 요청에 따라서 제33조만 요약해주세요.
"	"USER : “면직 사유에는 어떤 것들이 있나요?”
RAG :
다음은 서울시설공단 인사규정 중 제37조(직위해제) 및 제12조(결격사유) 전문입니다.

[제37조]
“① 이사장은 직원이 다음 각호의 1에 해당할 때에는 직위를 해제할 수 있다.

형사사건으로 공소가 제기된 자. 단, 약식명령이 청구된 자는 제외한다.(2005.6.30)

직무수행능력이 현저히 부족하거나 근무성적이 극히 불량한 자

소속 직원에 대한 감독능력이 부족한 자

징계등 의결요구 중인 자(개정 2018.5.21.)

공단 직원으로서 품위를 훼손하거나 현저히 공단 이익에 반한 행위를 한 자.”

[제12조]
“① 다음 각호의 1에 해당할 때에는 직원으로 채용할 수 없다.

피성년후견인 또는 피한정후견인(2015.3.31)

파산선고를 받고 복권되지 아니한 자(2015.3.31)

금고 이상의 형을 받고 그 집행이 종료되거나 집행받지 아니하기로 확정된 후 5년을 경과하지 아니한 자(2015.3.31)

금고 이상의 형을 받고 집행유예기간이 종료된 날로부터 2년을 경과하지 아니한 자(2015.3.31)

금고 이상의 형의 선고유예를 받은 자

「부패방지 및 국민권익위원회 설치 및 운영에 관한 법률」제82조에 따른 비위면직자 등(2013.12.9, 2015.3.31, 2017.7.25.)

법원 판결 또는 다른 법률에 따라 자격이 상실·정지된 자(2015.3.31)
7의2. 성폭력범죄 처벌 등에 관한 특례법 제2조에 따른 죄(100만원 이상 벌금형 확정 후 3년 미경과)

병역의무자로서 병역 기피 사실이 있는 자

정보통신망 이용촉진 및 정보보호 등에 관한 법률 제74조 제1항 제2·3호 위반 시 100만원 이상 벌금형 확정 후 3년 미경과.”

위 내용을 바탕으로, USER의 질문에 답변해주세요."	"USER : “수습임용 기간 중 해고 사유는 무엇인가요?”
RAG :
다음은 서울시설공단 인사규정 중 제11조(수습임용) 전문입니다.
“① 4급 이하 직원을 신규채용할 때는 3개월 미만의 수습기간을 둘 수 있다.(’03.11.7 단서삭제)<개정 2019.6.27>
② 신규채용된 자가 수습기간 중 다음 각호의 1에 해당할 때에는 채용하지 아니한다.

근무성적이 불량할 때

공단의 제규정을 위반하였을 때

제12조의 결격사유에 해당할 때.”

위 내용을 바탕으로, USER의 질문에 답변해주세요.
"


In [ ]:
"""
사용법
"""
chmod +x search_chunked.py


/home/조기정/project/RAG_LLM/src/test/search_chunked.py
python search_chunked.py "인사위원회 회의록 초안을 작성해줘." --top_k 5

In [ ]:
# conda create --name Qwen2.5 python=3.11.13
conda actiavate Qwen2.5
conda actiavate Qwen
cd /home/조기정/project/RAG_LLM



In [ ]:
"""
hugging 모델 다운로드 기능 
"""

#!/usr/bin/env python3
# download_llm_models.py

import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# 다운로드할 모델 리스트 (후보가 여러 개라면 여기에 추가)
MODEL_REPOS = [
    "Qwen/Qwen2.5-7B-Instruct-1M",
    # 예시: "Qwen/Qwen3-4B",
    #       "google/gemma-3n-E4B-it",
]

# 저장할 기본 경로
BASE_SAVE_DIR = "/home/조기정/project/RAG_LLM/src/test/llm_model"

def download_and_save_model(repo_id: str, save_dir: str):
    """
    repo_id Hugging Face 모델을 다운로드하여 save_dir에 저장한다.
    """
    print(f"▶ 다운로드 시작: {repo_id}")
    # 1) 토크나이저 로드 및 저장
    tokenizer = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    tokenizer.save_pretrained(save_dir)
    # 2) 모델 로드 및 저장
    model = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True)
    model.save_pretrained(save_dir)
    print(f"✔ 저장 완료: {save_dir}")

def main():
    os.makedirs(BASE_SAVE_DIR, exist_ok=True)

    for repo in MODEL_REPOS:
        # 모델명만 떼어내기 (슬래시 뒤 부분)
        model_name = repo.split("/")[-1]
        target_dir = os.path.join(BASE_SAVE_DIR, model_name)
        os.makedirs(target_dir, exist_ok=True)
        download_and_save_model(repo, target_dir)

if __name__ == "__main__":
    main()

▶ 스냅샷 다운로드 시작: Qwen/Qwen2.5-7B-Instruct-1M


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

{"timestamp":"2025-07-15T02:33:45.354844Z","level":"WARN","fields":{"message":"Reqwest(reqwest::Error { kind: Request, url: \"https://transfer.xethub.hf.co/xorbs/default/0a0886bed087c3f21a39dbf31fdd8254b61ce1e7c3b87e09efe9e587fcbebb9f?X-Xet-Signed-Range=bytes%3D0-57757005&Expires=1752550394&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC8wYTA4ODZiZWQwODdjM2YyMWEzOWRiZjMxZmRkODI1NGI2MWNlMWU3YzNiODdlMDllZmU5ZTU4N2ZjYmViYjlmP1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDAtNTc3NTcwMDUiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTI1NTAzOTR9fX1dfQ__&Signature=Ojb9xBJs0XpQQhhYTEYMOnzdUdngGBPx6h0iFBLbMdPezqhGMrOEduh42JJeuvOK22AFXqfATeBVpZ-i6JR4h0I6HZ00tbzLIvZTKr424mk9U2HRt5v~1RaGuPIDE86bam6YmKZCbeOhcjHJwYprktgqjQYeWm5NnKtRf6SwrLs~HcZBHDjx-V6DEHfDwnt6vXVv1LPqbaFd42~t1BBc9OG~SSPdmbxmCrTiAW-l2wURiaQBYjfIJ6gGfRWggSvfALec3bBxqXXyTlYRyrciwgDfNrWpUFXZ4XgEMhB1D4IpsmQRE3uMQNe3imgx1CMgMsv9uycIWwqaKVYEEwcR5A__&Key-Pair-Id=K2L8F4GPSG1IFC\",

Fetching 15 files:  47%|████▋     | 7/15 [2:04:26<2:22:12, 1066.61s/it]


In [ ]:
google/gemma-3n-E4B-it
unsloth/gemma-2-9b-it


In [7]:
"""
보안 레벨 검증

# FAISS GPU
pip uninstall faiss-cpu -y
pip install faiss-cpu
pip install faiss-gpu
"""
#!/usr/bin/env python3
# faiss_store.py

import os
import json
from pathlib import Path

import torch
import faiss
from transformers import AutoTokenizer, AutoModel

# 1) 설정
MODEL_DIR       = "/home/조기정/project/RAG_LLM/src/test/llm_model/Qwen2.5-7B-Instruct-1M"
TEXT_DIR        = Path("extracted_texts")
INDEX_PATH      = Path("faiss_index.idx")
META_PATH       = Path("faiss_metadata.json")
EMBED_DIM       = 512   # mean-pool 후 차원 (모델에 따라 다름)
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# 2) 토크나이저·모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model     = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True)
model.to(DEVICE).eval()

# 3) FAISS 인덱스 준비 (Inner Product + ID 매핑 + 삭제 가능)
flat_index = faiss.IndexFlatIP(EMBED_DIM)
index      = faiss.IndexIDMap2(flat_index)

# 4) 메타데이터 불러오기/초기화
if META_PATH.exists():
    with META_PATH.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    metadata = {}

# 5) 텍스트 임베딩 함수 (mean-pooling)
def embed_text(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # 마지막 레이어 hidden_states
    last_hs = outputs.hidden_states[-1].squeeze(0)  # (seq_len, dim)
    mask    = inputs["attention_mask"].unsqueeze(-1)
    sum_vec = (last_hs * mask).sum(0)
    cnt     = mask.sum()
    return (sum_vec / cnt).cpu().numpy()

# 6) 인덱스에 추가
next_id = max(map(int, metadata.keys()), default=0) + 1
for txt_path in TEXT_DIR.rglob("*.txt"):
    rel_path = str(txt_path.relative_to(TEXT_DIR))
    if rel_path in metadata:
        continue  # 이미 처리된 문서 건너뜀

    text = txt_path.read_text(encoding="utf-8")
    embedding = embed_text(text)

    # 예시: 보안레벨과 카테고리는 파일명이나 외부 매핑에서 결정
    security_level = "level_1"             # ← 실제 값으로 교체
    category       = rel_path.split("/")[0]  # 예: 상위 폴더명을 카테고리로

    # FAISS 삽입
    doc_id = next_id
    index.add_with_ids(embedding.reshape(1, -1), np.array([doc_id], dtype="int64"))

    # 메타데이터 저장
    metadata[doc_id] = {
        "path": rel_path,
        "security_level": security_level,
        "category": category
    }
    next_id += 1
    print(f"Added ID {doc_id}: {rel_path} [{security_level}, {category}]")

# 7) 인덱스·메타데이터 디스크에 저장
faiss.write_index(index, str(INDEX_PATH))
with META_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ FAISS index 및 metadata 저장 완료")


ValueError: Unrecognized model in /home/조기정/project/RAG_LLM/src/test/llm_model/Qwen2.5-7B-Instruct-1M. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v3, deformable_detr, deit, depth_anything, depth_pro, deta, detr, diffllama, dinat, dinov2, dinov2_with_registers, distilbert, donu